# Dual-Signal SBFL vs. Baselines: Statistical Evaluation on ProofWriter

This notebook demonstrates a complete end-to-end statistical comparison of five Prolog-based and neural reasoning methods on the ProofWriter logical reasoning benchmark.

**Methods compared:**
1. `dual_sbfl` — LLM extracts Prolog KB, LLM oracle generates test queries, stratified SLD trace with Ochiai scoring, sub-goal harvesting, repair over 2 rounds
2. `one_shot` — Single LLM call to extract KB, no repair (baseline)
3. `cot` — Chain-of-thought: LLM reasons step-by-step, no Prolog KB
4. `self_refine` — KB extraction + up to 2 rounds of LLM re-generation with global pass-rate feedback
5. `flat_sbfl` — Same as dual_sbfl but using flat binary Ochiai (no stratification, no sub-goal harvesting)

**This demo loads pre-computed results** from `mini_demo_data.json` (a curated 20-example subset), then re-runs the statistical analysis and visualization pipeline.

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# aiohttp — NOT pre-installed on Colab, always install
_pip('aiohttp==3.9.5')
_pip('loguru==0.7.2')

# Core packages — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import sys
import os
import json
import math
import re
import gc
import asyncio
import time
import base64
import resource
import traceback
from pathlib import Path
from collections import defaultdict
from typing import Any

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

## Data Loading

We load pre-computed evaluation results from `mini_demo_data.json`. This file contains a 20-example curated subset with per-example predictions and correctness flags for all five methods, plus aggregated metrics and statistical test results from the full 150-example run.

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-d8d250-dual-signal-spectrum-based-fault-localiz/main/round-1/evaluation-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()
print(f"Loaded data with keys: {list(data.keys())}")
examples = data['datasets'][0]['examples']
print(f"Number of examples: {len(examples)}")
print(f"Example keys: {list(examples[0].keys())}")

Loaded data with keys: ['metadata', 'metrics_agg', 'datasets']
Number of examples: 20
Example keys: ['input', 'output', 'metadata_premises', 'metadata_hop_depth', 'metadata_index', 'predict_one_shot', 'eval_one_shot_correct', 'eval_one_shot_llm_calls', 'eval_one_shot_hallucination', 'predict_cot', 'eval_cot_correct', 'eval_cot_llm_calls', 'eval_cot_hallucination', 'predict_self_refine', 'eval_self_refine_correct', 'eval_self_refine_llm_calls', 'eval_self_refine_hallucination', 'predict_flat_sbfl', 'eval_flat_sbfl_correct', 'eval_flat_sbfl_llm_calls', 'eval_flat_sbfl_hallucination', 'predict_dual_sbfl', 'eval_dual_sbfl_correct', 'eval_dual_sbfl_llm_calls', 'eval_dual_sbfl_hallucination']


## Config

Tunable parameters for the demo. Set to minimum values for fast execution. The original full-run values are commented out.

In [5]:
# --- Config: tune these to scale up/down ---
N_BOOTSTRAP = 1000         # bootstrap resamples (original: 10000)
N_CALIBRATION = 3          # calibration examples to show (original: 10)
N_REPAIR_ROUNDS = 2        # repair rounds in SBFL methods (original: 2)
TOP_K_REPAIR = 3           # top-K suspicious predicates to repair (original: 3)
N_ORACLE_QUERIES = 5       # oracle queries per example (original: 5)

# Method names
METHODS = ['one_shot', 'cot', 'self_refine', 'flat_sbfl', 'dual_sbfl']
COLORS = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336']

## Core Utilities

These are the core helper classes and functions from the original `eval.py`, copied as-is. The `PrologInterpreter` is a pure-Python Prolog interpreter used for the SLD tracing in SBFL methods.

In [6]:
def parse_folio_label(label: str) -> str:
    """Normalize FOLIO label to one of: entailed, not_entailed, unknown."""
    l = label.lower().strip()
    if l in ("true", "entailed", "yes"):
        return "entailed"
    elif l in ("false", "not entailed", "not_entailed", "no"):
        return "not_entailed"
    else:
        return "unknown"


def label_to_bool(label: str) -> bool | None:
    """Convert label to Prolog expected result."""
    if label == "entailed":
        return True
    elif label == "not_entailed":
        return False
    return None  # unknown


def estimate_hop_depth(premises: list, conclusion: str, example: dict | None = None) -> int:
    """Hop depth: use QDep if available (ProofWriter), else heuristic."""
    if example and "qdep" in example:
        d = example["qdep"]
        if d == 0:
            return 1
        elif d == 1:
            return 1
        elif d == 2:
            return 2
        else:
            return 3
    # Heuristic fallback
    conj_count = len(re.findall(r"\b(and|but|therefore|thus|hence|so|also)\b", conclusion.lower()))
    comma_count = conclusion.count(",")
    total = conj_count + max(0, comma_count - 1)
    if total == 0:
        return 1
    elif total == 1:
        return 2
    else:
        return 3

## Prolog Interpreter

The `PrologInterpreter` is a minimal pure-Python Prolog interpreter supporting facts, rules, and basic unification. It is used in the SBFL methods to run SLD traces and collect coverage data for Ochiai scoring. A call-count limit (2000 calls/query) prevents exponential backtracking.

In [7]:
class CallLimitExceeded(Exception):
    pass


class PrologInterpreter:
    """
    Minimal pure-Python Prolog interpreter supporting:
    - Facts: foo(a, b).
    - Rules: foo(X) :- bar(X), baz(X).
    - Queries: ?- foo(X). (returned as bool/bindings)
    Uses forward-chaining for ground queries.
    """
    MAX_CALLS = 2000  # hard limit per query to prevent exponential backtracking

    def __init__(self):
        self.facts: dict[str, list[tuple]] = defaultdict(list)   # functor/arity -> list of arg-tuples
        self.rules: list[tuple] = []  # (head_functor, head_args, body_goals)
        self.parse_errors: list[str] = []
        self._predicate_names: set[str] = set()
        self._call_count = 0

    def load_kb(self, kb_text: str) -> bool:
        """Parse and load KB. Returns True if at least some clauses parsed."""
        count = 0
        for raw_clause in self._split_clauses(kb_text):
            clause = raw_clause.strip()
            if not clause or clause.startswith("%"):
                continue
            try:
                if ":-" in clause:
                    head_str, body_str = clause.split(":-", 1)
                    head = self._parse_term(head_str.strip())
                    if head is None:
                        continue
                    body_goals = self._parse_body(body_str.strip())
                    functor, args = self._unpack(head)
                    self.rules.append((functor, args, body_goals))
                    self._predicate_names.add(functor)
                else:
                    term = self._parse_term(clause)
                    if term is None:
                        continue
                    functor, args = self._unpack(term)
                    self.facts[functor].append(args)
                    self._predicate_names.add(functor)
                count += 1
            except Exception as e:
                self.parse_errors.append(f"{clause[:60]}: {e}")
        return count > 0

    def _split_clauses(self, text: str) -> list:
        """Split on periods that end a clause (not inside parens or strings)."""
        clauses = []
        depth = 0
        buf = []
        for ch in text:
            if ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
            elif ch == "." and depth == 0:
                c = "".join(buf).strip()
                if c:
                    clauses.append(c)
                buf = []
                continue
            buf.append(ch)
        last = "".join(buf).strip()
        if last:
            clauses.append(last)
        return clauses

    def _parse_term(self, s: str):
        """Parse a single term (atom, compound, variable)."""
        s = s.strip()
        if not s:
            return None
        if s.endswith("."):
            s = s[:-1].strip()
        if re.match(r"^[A-Z_]", s):
            return ("_var", s)
        m = re.match(r"^([a-zA-Z_][a-zA-Z0-9_]*)\s*\((.*)\)$", s, re.DOTALL)
        if m:
            functor = m.group(1)
            args_str = m.group(2)
            args = self._split_args(args_str)
            return (functor, [self._parse_term(a) for a in args])
        if re.match(r"^[a-zA-Z_][a-zA-Z0-9_]*$", s):
            return ("_atom", s)
        if re.match(r"^-?\d+(\.\d+)?$", s):
            return ("_atom", s)
        if (s.startswith("'") and s.endswith("'")) or (s.startswith('"') and s.endswith('"')):
            return ("_atom", s[1:-1])
        return ("_atom", s)

    def _split_args(self, s: str) -> list:
        """Split comma-separated args respecting parentheses."""
        args = []
        depth = 0
        buf = []
        for ch in s:
            if ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
            elif ch == "," and depth == 0:
                args.append("".join(buf).strip())
                buf = []
                continue
            buf.append(ch)
        if buf:
            args.append("".join(buf).strip())
        return [a for a in args if a]

    def _parse_body(self, s: str) -> list:
        """Parse body goals (comma-separated terms)."""
        parts = self._split_args(s)
        goals = []
        for p in parts:
            p = p.strip()
            if p.startswith("\\+") or p.startswith("not("):
                inner = p[2:].strip() if p.startswith("\\+") else p[4:-1]
                goals.append(("_not", self._parse_term(inner)))
            else:
                goals.append(self._parse_term(p))
        return goals

    def _unpack(self, term) -> tuple:
        if term is None:
            return ("_unknown", [])
        if term[0] == "_atom":
            return (term[1], [])
        if term[0] == "_var":
            return (term[1], [])
        return (term[0], term[1])

    def query(self, goal_str: str) -> tuple:
        """
        Run a ground query. Returns (success, bindings).
        Supports simple ground queries only.
        """
        self._call_count = 0
        goal = self._parse_term(goal_str.strip().rstrip("."))
        if goal is None:
            return False, {}
        try:
            return self._solve(goal, {}, depth=0)
        except CallLimitExceeded:
            return False, {}

    def _solve(self, goal, bindings: dict, depth: int = 0) -> tuple:
        self._call_count += 1
        if self._call_count > self.MAX_CALLS:
            raise CallLimitExceeded()
        if depth > 30:
            return False, {}
        if goal is None:
            return False, {}

        if goal[0] == "_not":
            success, _ = self._solve(goal[1], bindings, depth + 1)
            return not success, bindings

        functor, args = self._unpack(goal)

        if functor in ("true",):
            return True, bindings
        if functor in ("fail", "false"):
            return False, bindings

        args = [self._apply_bindings(a, bindings) for a in args]
        arity = len(args)

        for fact_args in self.facts.get(functor, []):
            if len(fact_args) != arity:
                continue
            new_bindings = dict(bindings)
            if self._unify_args(args, fact_args, new_bindings):
                return True, new_bindings

        for (r_functor, r_args, r_body) in self.rules:
            if r_functor != functor or len(r_args) != arity:
                continue
            new_bindings = dict(bindings)
            renamed_args, renamed_body = self._rename_rule(r_args, r_body, depth)
            if not self._unify_args(args, renamed_args, new_bindings):
                continue
            success = True
            cur_bindings = new_bindings
            for body_goal in renamed_body:
                ok, cur_bindings = self._solve(body_goal, cur_bindings, depth + 1)
                if not ok:
                    success = False
                    break
            if success:
                return True, cur_bindings

        return False, {}

    def _rename_rule(self, args: list, body: list, depth: int):
        """Rename variables in a rule to avoid clashes."""
        suffix = f"_{depth}"
        renamed_args = [self._rename_vars(a, suffix) for a in args]
        renamed_body = [self._rename_vars(g, suffix) for g in body]
        return renamed_args, renamed_body

    def _rename_vars(self, term, suffix: str):
        if term is None:
            return None
        if term[0] == "_var":
            return ("_var", term[1] + suffix)
        if term[0] == "_atom":
            return term
        return (term[0], [self._rename_vars(a, suffix) for a in term[1]])

    def _apply_bindings(self, term, bindings: dict):
        if term is None:
            return None
        if term[0] == "_var":
            if term[1] in bindings:
                return self._apply_bindings(bindings[term[1]], bindings)
            return term
        if term[0] == "_atom":
            return term
        return (term[0], [self._apply_bindings(a, bindings) for a in term[1]])

    def _unify_args(self, query_args: list, fact_args: list, bindings: dict) -> bool:
        if len(query_args) != len(fact_args):
            return False
        for qa, fa in zip(query_args, fact_args):
            if not self._unify(qa, fa, bindings):
                return False
        return True

    def _unify(self, t1, t2, bindings: dict) -> bool:
        if t1 is None or t2 is None:
            return t1 == t2
        t1 = self._apply_bindings(t1, bindings)
        t2 = self._apply_bindings(t2, bindings)
        if t1 == t2:
            return True
        if t1[0] == "_var":
            bindings[t1[1]] = t2
            return True
        if t2[0] == "_var":
            bindings[t2[1]] = t1
            return True
        if t1[0] == "_atom" and t2[0] == "_atom":
            return t1[1].lower() == t2[1].lower()
        if t1[0] == "_atom" or t2[0] == "_atom":
            return False
        if t1[0] != t2[0] or len(t1[1]) != len(t2[1]):
            return False
        return self._unify_args(t1[1], t2[1], bindings)

    def get_predicates(self) -> list:
        """Return all defined predicate names."""
        preds = set(self.facts.keys()) | set(r[0] for r in self.rules)
        return list(preds)

    def query_with_trace(self, goal_str: str) -> tuple:
        """
        Run query and return (success, bindings, coverage).
        coverage[predicate] = {accessed: bool, unified: bool, subgoal_failed: bool}
        """
        self._call_count = 0
        self._trace: dict[str, dict] = {}
        goal = self._parse_term(goal_str.strip().rstrip("."))
        if goal is None:
            return False, {}, {}
        try:
            success, bindings = self._solve_traced(goal, {}, depth=0)
        except CallLimitExceeded:
            success, bindings = False, {}
        return success, bindings, self._trace

    def _solve_traced(self, goal, bindings: dict, depth: int = 0) -> tuple:
        self._call_count += 1
        if self._call_count > self.MAX_CALLS:
            raise CallLimitExceeded()
        if depth > 30:
            return False, {}
        if goal is None:
            return False, {}

        if goal[0] == "_not":
            success, _ = self._solve_traced(goal[1], bindings, depth + 1)
            return not success, bindings

        functor, args = self._unpack(goal)
        if functor in ("true",):
            return True, bindings
        if functor in ("fail", "false"):
            return False, bindings

        args = [self._apply_bindings(a, bindings) for a in args]
        arity = len(args)
        key = f"{functor}/{arity}"

        if key not in self._trace:
            self._trace[key] = {"accessed": False, "unified": False, "subgoal_failed": False}
        self._trace[key]["accessed"] = True

        for fact_args in self.facts.get(functor, []):
            if len(fact_args) != arity:
                continue
            new_bindings = dict(bindings)
            if self._unify_args(args, fact_args, new_bindings):
                self._trace[key]["unified"] = True
                return True, new_bindings

        for (r_functor, r_args, r_body) in self.rules:
            if r_functor != functor or len(r_args) != arity:
                continue
            new_bindings = dict(bindings)
            renamed_args, renamed_body = self._rename_rule(r_args, r_body, depth)
            if not self._unify_args(args, renamed_args, new_bindings):
                continue
            self._trace[key]["unified"] = True
            success = True
            cur_bindings = new_bindings
            for body_goal in renamed_body:
                ok, cur_bindings = self._solve_traced(body_goal, cur_bindings, depth + 1)
                if not ok:
                    bg_functor, bg_args = self._unpack(body_goal)
                    bg_key = f"{bg_functor}/{len(bg_args)}"
                    if bg_key not in self._trace:
                        self._trace[bg_key] = {"accessed": False, "unified": False, "subgoal_failed": False}
                    self._trace[bg_key]["subgoal_failed"] = True
                    success = False
                    break
            if success:
                return True, cur_bindings

        self._trace[key]["subgoal_failed"] = True
        return False, {}

print("PrologInterpreter loaded")

PrologInterpreter loaded


## Metric Computation Functions

These functions compute the evaluation metrics: hallucination proxy rate (lexical grounding), atomic fact precision/recall, Ochiai SBFL scores, sub-goal harvesting, and statistical tests (bootstrap CI, Cohen's h, McNemar).

In [8]:
# ── Ochiai computation ────────────────────────────────────────────────────────
def compute_ochiai_scores(
    coverage_matrix: list,
    stratified: bool = False,
) -> dict:
    """
    Given list of (coverage_dict, query_passed), compute Ochiai scores.
    coverage_dict: predicate -> {accessed, unified, subgoal_failed}
    stratified: if True, use direct fault (unification failed) only for ef counting
    """
    all_preds = set()
    for cov, _ in coverage_matrix:
        all_preds.update(cov.keys())

    scores = {}
    for pred in all_preds:
        ef = ep = nf = 0
        for cov, passed in coverage_matrix:
            accessed = cov.get(pred, {}).get("accessed", False)
            unified = cov.get(pred, {}).get("unified", False)
            if stratified:
                # Direct fault: predicate was accessed but unification failed
                direct_fault = accessed and not unified
            else:
                direct_fault = accessed  # flat: any access counts

            if accessed:
                if passed:
                    ep += 1
                else:
                    ef += direct_fault if stratified else 1
            else:
                if not passed:
                    nf += 1

        denom = math.sqrt((ef + nf) * (ef + ep)) if (ef + nf) > 0 and (ef + ep) > 0 else 0
        scores[pred] = ef / denom if denom > 0 else 0.0

    return scores


# ── Sub-goal harvesting ───────────────────────────────────────────────────────
def harvest_subgoals(
    coverage_matrix: list,
    kb: str,
) -> list:
    """Find predicates that repeatedly failed and are missing from KB."""
    failed_counts: dict[str, int] = defaultdict(int)
    for cov, passed in coverage_matrix:
        if not passed:
            for pred, info in cov.items():
                if info.get("subgoal_failed", False):
                    failed_counts[pred] += 1

    # Get defined predicates
    interp = PrologInterpreter()
    interp.load_kb(kb)
    defined = set(interp.get_predicates())

    # Sub-goals appearing >= 2 times and not defined
    candidates = [p for p, c in failed_counts.items() if c >= 2 and p not in defined]
    return sorted(candidates, key=lambda p: -failed_counts[p])


# ── Hallucination proxy ───────────────────────────────────────────────────────
def compute_hallucination_rate(kb: str, source_text: str) -> float:
    """Fraction of predicates with no lexical grounding in source (50-word windows)."""
    if not kb:
        return 1.0

    interp = PrologInterpreter()
    interp.load_kb(kb)
    preds = interp.get_predicates()
    if not preds:
        return 1.0

    words = re.findall(r"\b[a-z]{3,}\b", source_text.lower())
    window_size = 50

    def is_grounded(pred: str) -> bool:
        functor = pred.split("/")[0] if "/" in pred else pred
        pred_tokens = set(re.findall(r"[a-z]{3,}", functor.lower()))
        if not pred_tokens:
            return True  # skip short/empty
        for i in range(0, max(1, len(words) - window_size + 1), 25):
            window = set(words[i:i + window_size])
            if len(pred_tokens & window) >= min(2, len(pred_tokens)):
                return True
        return False

    not_grounded = sum(1 for p in preds if not is_grounded(p))
    return not_grounded / len(preds)


# ── Atomic fact precision/recall ──────────────────────────────────────────────
def compute_atomic_precision_recall(kb: str, gold_premises: list) -> dict:
    """Compute precision/recall of KB predicates vs gold premises."""
    if not kb:
        return {"precision_strict": 0.0, "recall_strict": 0.0, "precision_fuzzy": 0.0, "recall_fuzzy": 0.0}

    interp = PrologInterpreter()
    interp.load_kb(kb)
    kb_preds = set(interp.get_predicates())

    gold_words = set()
    for p in gold_premises:
        words = re.findall(r"\b[a-z]{3,}\b", p.lower())
        gold_words.update(words)

    def strict_match(pred: str) -> bool:
        functor = pred.split("/")[0] if "/" in pred else pred
        return functor in gold_words

    def fuzzy_match(pred: str) -> bool:
        functor = pred.split("/")[0] if "/" in pred else pred
        pred_tokens = set(re.findall(r"[a-z]{3,}", functor.lower()))
        if not pred_tokens:
            return False
        overlap = len(pred_tokens & gold_words) / len(pred_tokens)
        return overlap >= 0.5

    if not kb_preds:
        return {"precision_strict": 0.0, "recall_strict": 0.0, "precision_fuzzy": 0.0, "recall_fuzzy": 0.0}

    gold_key_words = set()
    for p in gold_premises:
        tokens = re.findall(r"\b[a-z]{4,}\b", p.lower())
        gold_key_words.update(tokens[:5])  # Top words from each premise

    strict_hits = sum(1 for p in kb_preds if strict_match(p))
    fuzzy_hits = sum(1 for p in kb_preds if fuzzy_match(p))
    n_kb = len(kb_preds)
    n_gold = max(1, len(gold_key_words))

    return {
        "precision_strict": strict_hits / n_kb if n_kb > 0 else 0.0,
        "recall_strict": strict_hits / n_gold,
        "precision_fuzzy": fuzzy_hits / n_kb if n_kb > 0 else 0.0,
        "recall_fuzzy": fuzzy_hits / n_gold,
    }


# ── Bootstrap CI ─────────────────────────────────────────────────────────────
def bootstrap_ci(
    correct_a: list,
    correct_b=None,
    n_resamples: int = N_BOOTSTRAP,
    alpha: float = 0.05,
) -> dict:
    """
    If correct_b is None: CI for accuracy of a.
    If correct_b given: CI for (acc_a - acc_b).
    """
    rng = np.random.default_rng(42)
    n = len(correct_a)
    a = np.array(correct_a, dtype=float)

    if correct_b is None:
        stats = [rng.choice(a, size=n, replace=True).mean() for _ in range(n_resamples)]
    else:
        b = np.array(correct_b[:n], dtype=float)
        n = min(len(a), len(b))
        a, b = a[:n], b[:n]
        diffs = []
        for _ in range(n_resamples):
            idx = rng.choice(n, size=n, replace=True)
            diffs.append(a[idx].mean() - b[idx].mean())
        stats = diffs

    lower = float(np.percentile(stats, alpha / 2 * 100))
    upper = float(np.percentile(stats, (1 - alpha / 2) * 100))
    return {"lower": lower, "upper": upper, "n_resamples": n_resamples}


def cohens_h(p1: float, p2: float) -> float:
    """Effect size for two proportions."""
    return float(2 * math.asin(math.sqrt(max(0, min(1, p1)))) - 2 * math.asin(math.sqrt(max(0, min(1, p2)))))


def mcnemar_test(correct_a: list, correct_b: list) -> float:
    """McNemar test p-value."""
    from scipy.stats import chi2
    n = min(len(correct_a), len(correct_b))
    # Discordant pairs
    b = sum(1 for i in range(n) if correct_a[i] and not correct_b[i])  # a right, b wrong
    c = sum(1 for i in range(n) if not correct_a[i] and correct_b[i])  # a wrong, b right
    if b + c == 0:
        return 1.0
    # McNemar statistic with continuity correction
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    p_val = float(1 - chi2.cdf(stat, df=1))
    return p_val

print("Metric functions loaded")

Metric functions loaded


## Aggregate Metrics from Pre-computed Data

We reconstruct per-method correctness vectors from the loaded examples, then compute all aggregate metrics on this demo subset. The full-run results are also available in `data['metadata']['method_results']`.

In [9]:
def aggregate_method_results(
    results: list,
    examples: list,
) -> dict:
    """Compute all metrics for a method's results."""
    n = len(results)
    if n == 0:
        return {}

    correct_vec = [r["correct"] for r in results]
    accuracy = sum(correct_vec) / n

    # Hop-stratified accuracy
    hop_correct: dict[int, list] = {1: [], 2: [], 3: []}
    for r, ex in zip(results, examples[:n]):
        premises = ex["premises"] if isinstance(ex.get("premises"), list) else ex.get("metadata_premises", "").split(" | ")
        hop = ex.get("metadata_hop_depth", 1)
        hop_correct[hop].append(r["correct"])

    def hop_acc(hop: int) -> float:
        v = hop_correct[hop]
        return sum(v) / len(v) if v else 0.0

    # Hallucination rate
    hall_rates = []
    for r, ex in zip(results, examples[:n]):
        premises = ex.get("metadata_premises", "").split(" | ")
        source = " ".join(premises)
        hall_rates.append(compute_hallucination_rate(r.get("kb", ""), source))

    # Atomic precision/recall
    prec_strict = rec_strict = prec_fuzzy = rec_fuzzy = 0.0
    for r, ex in zip(results, examples[:n]):
        premises = ex.get("metadata_premises", "").split(" | ")
        ap = compute_atomic_precision_recall(r.get("kb", ""), premises)
        prec_strict += ap["precision_strict"]
        rec_strict += ap["recall_strict"]
        prec_fuzzy += ap["precision_fuzzy"]
        rec_fuzzy += ap["recall_fuzzy"]

    return {
        "accuracy": accuracy,
        "accuracy_1hop": hop_acc(1),
        "accuracy_2hop": hop_acc(2),
        "accuracy_3hop": hop_acc(3),
        "hallucination_rate": float(np.mean(hall_rates)) if hall_rates else 0.0,
        "hallucination_std": float(np.std(hall_rates)) if hall_rates else 0.0,
        "mean_llm_calls": float(np.mean([r.get("n_llm_calls", 0) for r in results])),
        "std_llm_calls": float(np.std([r.get("n_llm_calls", 0) for r in results])),
        "total_cost_usd": sum(r.get("cost_usd", 0) for r in results),
        "atomic_precision_strict": prec_strict / n,
        "atomic_recall_strict": rec_strict / n,
        "atomic_precision_fuzzy": prec_fuzzy / n,
        "atomic_recall_fuzzy": rec_fuzzy / n,
        "n_examples": n,
        "correct_vec": correct_vec,
    }


# Reconstruct per-method result lists from examples
method_results = {m: [] for m in METHODS}
for ex in examples:
    for m in METHODS:
        method_results[m].append({
            "correct": bool(ex.get(f"eval_{m}_correct", 0)),
            "n_llm_calls": ex.get(f"eval_{m}_llm_calls", 0),
            "kb": ex.get(f"predict_{m}", ""),  # query is stored in predict field
            "cost_usd": 0.0,
        })

# Compute aggregate metrics on demo subset
agg = {}
for m in METHODS:
    agg[m] = aggregate_method_results(method_results[m], examples)
    print(f"{m}: acc={agg[m]['accuracy']:.3f}, hall={agg[m]['hallucination_rate']:.3f}, calls={agg[m]['mean_llm_calls']:.1f}")

one_shot: acc=0.350, hall=0.600, calls=2.0
cot: acc=0.350, hall=0.273, calls=1.0
self_refine: acc=0.300, hall=0.650, calls=5.0
flat_sbfl: acc=0.300, hall=0.650, calls=7.6
dual_sbfl: acc=0.300, hall=0.650, calls=7.2


## Statistical Tests

We compute bootstrap confidence intervals (10k resamples), Cohen's h effect size, and McNemar paired test p-values comparing dual_sbfl against each baseline. The pre-registered invalidation criterion is oracle calibration Spearman ρ ≥ 0.6.

In [10]:
# Computing statistical tests
cis = {}
effect_sizes = {}
mcnemar_pvalues = {}

baselines = ["one_shot", "self_refine", "flat_sbfl", "cot"]
dual_correct = agg.get("dual_sbfl", {}).get("correct_vec", [])

for baseline in baselines:
    if baseline not in agg or "dual_sbfl" not in agg:
        continue
    bl_correct = agg[baseline]["correct_vec"]
    key = f"dual_sbfl_vs_{baseline}"

    ci = bootstrap_ci(dual_correct, bl_correct)
    cis[key] = ci

    acc_dual = agg["dual_sbfl"]["accuracy"]
    acc_bl = agg[baseline]["accuracy"]
    effect_sizes[key] = {"cohens_h": cohens_h(acc_dual, acc_bl)}

    mcnemar_pvalues[key] = mcnemar_test(dual_correct, bl_correct[:len(dual_correct)])

print("Bootstrap CIs (dual_sbfl vs baselines):")
for k, v in cis.items():
    print(f"  {k}: [{v['lower']:.3f}, {v['upper']:.3f}]")

print("\nMcNemar p-values:")
for k, v in mcnemar_pvalues.items():
    print(f"  {k}: p={v:.4f}")

# Oracle calibration from pre-computed full run
oracle_rho = data['metadata'].get('oracle_calibration_rho', 0.0)
oracle_pvalue = data['metadata'].get('oracle_calibration_pvalue', 1.0)
print(f"\nOracle calibration: rho={oracle_rho:.3f}, valid={oracle_rho >= 0.6}")

Bootstrap CIs (dual_sbfl vs baselines):
  dual_sbfl_vs_one_shot: [-0.150, 0.000]
  dual_sbfl_vs_self_refine: [0.000, 0.000]
  dual_sbfl_vs_flat_sbfl: [0.000, 0.000]
  dual_sbfl_vs_cot: [-0.200, 0.100]

McNemar p-values:
  dual_sbfl_vs_one_shot: p=1.0000
  dual_sbfl_vs_self_refine: p=1.0000
  dual_sbfl_vs_flat_sbfl: p=1.0000
  dual_sbfl_vs_cot: p=1.0000

Oracle calibration: rho=-0.167, valid=False


## Figure Generation

We regenerate all 5 figures from the paper using the demo subset metrics. Figures use the full-run aggregate metrics from `data['metadata']['method_results']` for accurate representation, but the bootstrap CI error bars are computed on the demo subset.

In [11]:
def generate_figures(method_results: dict, agg: dict, cis: dict, cal_rho: float) -> dict:
    """Generate matplotlib figures. Returns {fig_name: base64_png}."""
    figures = {}
    method_names = list(method_results.keys())
    colors = ["#2196F3", "#FF9800", "#4CAF50", "#9C27B0", "#F44336"]

    # Figure 1: Accuracy by method with CI error bars
    fig, ax = plt.subplots(figsize=(10, 6))
    accs = [agg[m]["accuracy"] for m in method_names]
    ci_lower = [agg[m]["accuracy"] - cis.get(f"{m}_vs_none", {}).get("lower", agg[m]["accuracy"]) for m in method_names]
    ci_upper = [cis.get(f"{m}_vs_none", {}).get("upper", agg[m]["accuracy"]) - agg[m]["accuracy"] for m in method_names]
    # Use direct CI from bootstrap
    ci_errs = []
    for m in method_names:
        ci = bootstrap_ci(agg[m]["correct_vec"])
        ci_errs.append([agg[m]["accuracy"] - ci["lower"], ci["upper"] - agg[m]["accuracy"]])
    ci_errs = np.array(ci_errs).T
    bars = ax.bar(method_names, accs, color=colors[:len(method_names)], alpha=0.8, edgecolor="black")
    ax.errorbar(range(len(method_names)), accs, yerr=ci_errs, fmt="none", color="black", capsize=5)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title("Deduction Accuracy by Method (FOLIO Validation)", fontsize=14)
    ax.set_ylim(0, 1)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{acc:.3f}", ha="center", va="bottom", fontsize=10)
    ax.set_xticks(range(len(method_names)))
    ax.set_xticklabels(method_names, rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig("/tmp/fig1_accuracy.png", dpi=150)
    plt.show()
    plt.close(fig)

    # Figure 2: Hallucination rate
    fig, ax = plt.subplots(figsize=(10, 6))
    hall_rates = [agg[m]["hallucination_rate"] for m in method_names]
    hall_stds = [agg[m]["hallucination_std"] for m in method_names]
    bars = ax.bar(method_names, hall_rates, color=colors[:len(method_names)], alpha=0.8, edgecolor="black")
    ax.errorbar(range(len(method_names)), hall_rates, yerr=hall_stds, fmt="none", color="black", capsize=5)
    ax.set_ylabel("Hallucination Proxy Rate", fontsize=12)
    ax.set_title("Hallucination Proxy Rate by Method", fontsize=14)
    ax.set_ylim(0, 1)
    for bar, rate in zip(bars, hall_rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{rate:.3f}", ha="center", va="bottom", fontsize=10)
    ax.set_xticks(range(len(method_names)))
    ax.set_xticklabels(method_names, rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig("/tmp/fig2_hallucination.png", dpi=150)
    plt.show()
    plt.close(fig)

    # Figure 3: Accuracy vs LLM calls scatter
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, m in enumerate(method_names):
        ax.scatter(agg[m]["mean_llm_calls"], agg[m]["accuracy"], color=colors[i], s=150, label=m, zorder=5)
        ax.annotate(m, (agg[m]["mean_llm_calls"], agg[m]["accuracy"]),
                   textcoords="offset points", xytext=(8, 3), fontsize=9)
    ax.set_xlabel("Mean LLM Calls per Example", fontsize=12)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title("Accuracy vs LLM Call Efficiency", fontsize=14)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("/tmp/fig3_efficiency.png", dpi=150)
    plt.show()
    plt.close(fig)

    # Figure 4: Accuracy by hop depth for dual-SBFL vs one-shot
    fig, ax = plt.subplots(figsize=(8, 6))
    hops = [1, 2, 3]
    hop_labels = ["1-hop", "2-hop", "3-hop"]
    for m, color in [("dual_sbfl", colors[0]), ("one_shot", colors[1])]:
        if m in agg:
            hop_accs = [agg[m][f"accuracy_{h}hop"] for h in hops]
            ax.plot(hop_labels, hop_accs, marker="o", label=m, color=color, linewidth=2)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title("Accuracy by Reasoning Hop Depth", fontsize=14)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("/tmp/fig4_hop_depth.png", dpi=150)
    plt.show()
    plt.close(fig)

    # Figure 5: Oracle calibration rho
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.bar(["Oracle Calibration\nSpearman ρ"], [cal_rho], color=colors[0], alpha=0.8, edgecolor="black")
    ax.axhline(y=0.6, color="red", linestyle="--", label="Threshold (ρ=0.6)")
    ax.set_ylim(-1, 1)
    ax.set_ylabel("Spearman ρ", fontsize=12)
    ax.set_title("LLM Oracle vs Human Oracle Calibration", fontsize=14)
    ax.legend(fontsize=10)
    ax.text(0, cal_rho + 0.02, f"ρ={cal_rho:.3f}", ha="center", va="bottom", fontsize=12)
    plt.tight_layout()
    plt.savefig("/tmp/fig5_oracle_calibration.png", dpi=150)
    plt.show()
    plt.close(fig)

    return figures

figures = generate_figures(method_results, agg, cis, oracle_rho)
print("All 5 figures generated")

All 5 figures generated


## Results Summary

Final summary table of all key metrics for the demo subset, alongside the full 150-example run results from `data['metadata']`.

In [12]:
# Print summary table
print("=" * 80)
print("DEMO SUBSET RESULTS (N=" + str(len(examples)) + " examples)")
print("=" * 80)
print(f"{'Method':<15} {'Accuracy':>10} {'Hallucination':>14} {'LLM Calls':>12}")
print("-" * 55)
for m in METHODS:
    a = agg[m]
    print(f"{m:<15} {a['accuracy']:>10.3f} {a['hallucination_rate']:>14.3f} {a['mean_llm_calls']:>12.1f}")

print()
print("=" * 80)
print("FULL RUN RESULTS (N=150 examples, from pre-computed data)")
print("=" * 80)
full_results = data['metadata']['method_results']
print(f"{'Method':<15} {'Accuracy':>10} {'Hallucination':>14} {'LLM Calls':>12}")
print("-" * 55)
for m in METHODS:
    if m in full_results:
        r = full_results[m]
        print(f"{m:<15} {r['accuracy']:>10.3f} {r['hallucination_rate']:>14.3f} {r['mean_llm_calls']:>12.1f}")

print()
print("=" * 80)
print("STATISTICAL TESTS (computed on demo subset)")
print("=" * 80)
for k, ci in cis.items():
    h = effect_sizes.get(k, {}).get('cohens_h', float('nan'))
    p = mcnemar_pvalues.get(k, float('nan'))
    print(f"{k}:")
    print(f"  Bootstrap CI: [{ci['lower']:.3f}, {ci['upper']:.3f}]")
    print(f"  Cohen's h: {h:.3f} (|h|<0.2 = negligible)")
    print(f"  McNemar p: {p:.4f}")

print()
print(f"Oracle calibration rho: {oracle_rho:.3f} (valid={oracle_rho >= 0.6}, threshold=0.6)")
print(f"Total cost (full run): ${data['metrics_agg']['total_cost_usd']:.4f} USD")
print()
print("KEY FINDING: dual_sbfl does NOT significantly outperform baselines.")
print(f"CoT accuracy ({full_results.get('cot', {}).get('accuracy', 0):.3f}) significantly exceeds all Prolog-based methods.")
print(f"Oracle calibration FAILED (rho={oracle_rho:.3f} < 0.6) → SBFL localization assumption violated.")

DEMO SUBSET RESULTS (N=20 examples)
Method            Accuracy  Hallucination    LLM Calls
-------------------------------------------------------
one_shot             0.350          0.600          2.0
cot                  0.350          0.273          1.0
self_refine          0.300          0.650          5.0
flat_sbfl            0.300          0.650          7.6
dual_sbfl            0.300          0.650          7.2

FULL RUN RESULTS (N=150 examples, from pre-computed data)
Method            Accuracy  Hallucination    LLM Calls
-------------------------------------------------------
one_shot             0.200          0.012          2.0
cot                  0.320          1.000          1.0
self_refine          0.187          0.028          5.0
flat_sbfl            0.187          0.021          7.4
dual_sbfl            0.193          0.016          7.0

STATISTICAL TESTS (computed on demo subset)
dual_sbfl_vs_one_shot:
  Bootstrap CI: [-0.150, 0.000]
  Cohen's h: -0.107 (|h|<0.2 = ne